#### Fine-tune siêu tham số (Optuna) cho model GIÁ CƠ BẢN

Từ đầu tới giờ, HistGB/LightGBM/XGBoost đều dùng siêu tham số **chọn tay** (không search hệ thống).
Notebook này dùng **Optuna** (Bayesian optimization — thông minh hơn grid search thô, tự học vùng
tham số tốt qua từng vòng thử) để tìm bộ tham số tối ưu cho **LightGBM** (đại diện, vì tốc độ nhanh
nhất trong 3 thuật toán — kết quả 3 thuật toán đã chứng minh gần như giống hệt nhau nên không cần
tune riêng cả 3).

**Dùng đúng tập `validation`** đã tạo sẵn ở `chuan_bi_du_lieu.ipynb` nhưng **chưa từng dùng tới**
(đúng mục đích ban đầu: "Chỉnh siêu tham số") — train trên `train`, chọn tham số tốt nhất dựa trên
`validation`, rồi mới đánh giá cuối cùng trên `test` (không đụng test khi tìm tham số, tránh leak).

⚠️ **Lưu ý:** đã có 7 hướng thử trước đều xác nhận MAE ~15k VND là **sàn nhiễu** của dữ liệu — fine-tune
khó có khả năng vượt qua được, nhưng vẫn nên làm để **chứng minh đầy đủ**, không bỏ sót hướng nào.

**1. Nạp dữ liệu (đủ cả train/validation/test)**

In [1]:
import warnings, time
from pathlib import Path
import numpy as np, pandas as pd
import lightgbm as lgb
import optuna
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.width", 220)

CAT = ["service_name", "pickup_location_name", "dropoff_location_name", "weather_main"]
B_NUM = ["quote_distance", "quote_duration", "gio_vn", "latest_observed_base",
         "history_60m_price_mean", "history_60m_price_std", "history_60m_price_slope_per_minute",
         "latest_observed_quote_distance", "latest_observed_quote_duration", "actual_observation_age_minutes"]

PREP = Path("../../data/hcm_train_ready.parquet")
assert PREP.exists(), "Chua co hcm_train_ready.parquet -> chay chuan_bi_du_lieu.ipynb truoc!"
COLS = list(dict.fromkeys(CAT + B_NUM + ["target_shown_price", "target_shown_multiplier",
        "latest_observed_price", "latest_observed_multiplier", "evaluation_month", "split"]))
COLS = [c for c in COLS if c != "latest_observed_base"]
df = pd.read_parquet(PREP, columns=COLS)
df["base_price"] = df.target_shown_price / df.target_shown_multiplier.clip(lower=0.1)
df["latest_observed_base"] = df.latest_observed_price / df.latest_observed_multiplier.clip(lower=0.1)
for cc in CAT: df[cc] = df[cc].astype("category")
print(f"Nap {len(df):,} dong | split: {df.split.value_counts().to_dict()}")

Nap 6,897,051 dong | split: {'train': 4641799, 'test': 864360, 'validation': 774984, 'calibration': 615908}


**2. Định nghĩa vùng tìm kiếm (search space) + hàm mục tiêu (objective)**

Optuna thử nhiều bộ tham số, mỗi lần **train trên `train`, chấm điểm trên `validation`** (MAE) —
tự học dần vùng tham số tốt (khác grid search thử đều tất cả, Optuna ưu tiên thử vùng có khả năng tốt).

In [2]:
def objective(trial, tr, va):
    params = dict(
        n_estimators=trial.suggest_int("n_estimators", 200, 1200, step=100),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        num_leaves=trial.suggest_int("num_leaves", 15, 255, log=True),
        max_depth=trial.suggest_int("max_depth", 3, 12),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 200, log=True),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    )
    m = lgb.LGBMRegressor(**params, random_state=42, verbose=-1)
    m.fit(tr[CAT+B_NUM], np.log(tr.base_price),
          eval_set=[(va[CAT+B_NUM], np.log(va.base_price))],
          callbacks=[lgb.early_stopping(30, verbose=False)])
    pred = np.exp(m.predict(va[CAT+B_NUM]))
    return mean_absolute_error(va.base_price, pred)

print("Da dinh nghia objective function.")

Da dinh nghia objective function.


**3. Chạy Optuna theo từng tháng (40 trial/tháng)**

In [3]:
thangs = sorted(df.evaluation_month.unique())
studies, best_params = {}, {}
for th in thangs:
    sub = df[df.evaluation_month==th]
    tr = sub[sub.split=="train"]; va = sub[sub.split=="validation"]
    t0 = time.time()
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(lambda trial: objective(trial, tr, va), n_trials=40, show_progress_bar=False)
    studies[th] = study; best_params[th] = study.best_params
    print(f"  [{th}] best MAE (validation) = {study.best_value:,.0f} VND | {time.time()-t0:.1f}s")
    print(f"       best params: {study.best_params}")

  [2026-01] best MAE (validation) = 14,954 VND | 294.9s
       best params: {'n_estimators': 800, 'learning_rate': 0.017168190067971722, 'num_leaves': 52, 'max_depth': 6, 'min_child_samples': 27, 'subsample': 0.8919142340329773, 'colsample_bytree': 0.8347129912720158, 'reg_alpha': 0.09646659430657006, 'reg_lambda': 0.27571789093924465}
  [2026-02] best MAE (validation) = 14,978 VND | 316.6s
       best params: {'n_estimators': 1200, 'learning_rate': 0.04874893951830606, 'num_leaves': 39, 'max_depth': 12, 'min_child_samples': 39, 'subsample': 0.9538323976412588, 'colsample_bytree': 0.8721508551765098, 'reg_alpha': 0.251140750149761, 'reg_lambda': 6.688509637815339}
  [2026-03] best MAE (validation) = 15,056 VND | 243.0s
       best params: {'n_estimators': 1200, 'learning_rate': 0.056105998777192946, 'num_leaves': 34, 'max_depth': 12, 'min_child_samples': 39, 'subsample': 0.8211770829222278, 'colsample_bytree': 0.9805126451489784, 'reg_alpha': 0.2522044219097959, 'reg_lambda': 0.6873580

**4. So sánh: model TUNED vs BASELINE (tham số mặc định) — trên tập `test`**

In [4]:
def metrics(y, p):
    y, p = np.asarray(y), np.asarray(p)
    return dict(MAE=mean_absolute_error(y,p), RMSE=mean_squared_error(y,p)**.5,
                R2=r2_score(y,p), MAPE=np.mean(np.abs((y-p)/y))*100)

BASELINE_PARAMS = dict(n_estimators=800, learning_rate=0.03, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, random_state=42, verbose=-1)

rows = []
for th in thangs:
    sub = df[df.evaluation_month==th]
    tr = sub[sub.split=="train"]; te = sub[sub.split=="test"]

    m_base = lgb.LGBMRegressor(**BASELINE_PARAMS).fit(tr[CAT+B_NUM], np.log(tr.base_price))
    pred_base = np.exp(m_base.predict(te[CAT+B_NUM]))

    m_tuned = lgb.LGBMRegressor(**best_params[th], random_state=42, verbose=-1).fit(tr[CAT+B_NUM], np.log(tr.base_price))
    pred_tuned = np.exp(m_tuned.predict(te[CAT+B_NUM]))

    rows.append({"Thang": th, "Cach": "Baseline (tham so mac dinh)", "n": len(te), **metrics(te.base_price, pred_base)})
    rows.append({"Thang": th, "Cach": "Tuned (Optuna)", "n": len(te), **metrics(te.base_price, pred_tuned)})

bang = pd.DataFrame(rows).round(2)
print("SO SANH TUNED vs BASELINE (tren test, tung thang):")
display(bang)

tong = bang.groupby("Cach")[["MAE","MAPE"]].mean().round(2)
print("\nTRUNG BINH 3 THANG:")
display(tong)
diff = tong.loc["Tuned (Optuna)","MAE"] - tong.loc["Baseline (tham so mac dinh)","MAE"]
print(f"\nChenh lech MAE trung binh (Tuned - Baseline) = {diff:+,.0f} VND")

SO SANH TUNED vs BASELINE (tren test, tung thang):


,Thang,Cach,n,MAE,RMSE,R2,MAPE
0,2026-01,Baseline (tham so mac dinh),315360,15072.04,20160.04,0.66,14.61
1,2026-01,Tuned (Optuna),315360,15062.61,20149.24,0.66,14.60
2,2026-02,Baseline (tham so mac dinh),234632,14975.39,19933.45,0.66,14.57
3,2026-02,Tuned (Optuna),234632,14984.40,19947.00,0.66,14.58
4,2026-03,Baseline (tham so mac dinh),314368,15049.06,20095.70,0.65,14.59
5,2026-03,Tuned (Optuna),314368,15055.27,20103.51,0.65,14.60



TRUNG BINH 3 THANG:


,MAE,MAPE
Cach,,
Baseline (tham so mac dinh),15032.16,14.59
Tuned (Optuna),15034.09,14.59



Chenh lech MAE trung binh (Tuned - Baseline) = +2 VND


**Kết luận**

- Nếu Tuned thắng rõ Baseline → nên cập nhật `_common_train.py` (`tao_lightgbm()`) theo tham số tối ưu tìm được
- Nếu gần bằng/không đổi → xác nhận thêm 1 lần nữa: **đã chạm sàn nhiễu**, tinh chỉnh tham số không
  còn dư địa cải thiện (khớp với 7 hướng thử trước: đổi thuật toán, đổi loss, thêm feature, đổi target,
  chuẩn hóa theo tuyến, ném hết feature, GAM — tất cả đều không cải thiện được).